# Soup Kernel Lab — independent experiments

Repository: https://github.com/kok-o/soup-kernel-lab

This notebook fetches live source from the selected Git ref; it does not use the embedded hackathon snapshot.
The historical 61.40 s /729 MiB result belongs only to the preserved baseline.

1. Enable Internet. For training choose Tesla T4 GPU, then set `RUN_TRAINING = True`.
2. Set `REF` to your pushed branch, tag, or full commit SHA. `main` uses the latest pushed main when this session starts.
3. Use a fresh session and Run All. The resolved SHA and source hashes are saved with the results.

The initial saved version is a CPU checkout check (`RUN_TRAINING = False`); it does not train or consume GPU quota.
The old competition notebook is unchanged.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, hashlib
REPOSITORY = 'https://github.com/kok-o/soup-kernel-lab.git'
REF = 'main'  # e.g. 'experiment/head-chunks' or a full commit SHA
RUN_TRAINING = False  # enable only after selecting a T4 accelerator
WORKSPACE = Path('/kaggle/working')
ROOT = WORKSPACE / 'soup-kernel-lab'
def run(args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)
assert REF and not REF.startswith('-'), 'Use a branch, tag, or commit SHA'
assert not ROOT.exists(), 'Start a fresh session; an existing checkout is never overwritten.'
run(['git', 'init', str(ROOT)])
run(['git', '-C', str(ROOT), 'remote', 'add', 'origin', REPOSITORY])
run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', REF])
run(['git', '-C', str(ROOT), 'checkout', '--detach', 'FETCH_HEAD'])
COMMIT = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
tracked = subprocess.check_output(['git', '-C', str(ROOT), 'ls-files', '-z', '--', 'solution', 'harness', 'task', 'tests']).decode().split('\0')
def digest(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()
EXPECTED = {name: digest(ROOT / name) for name in tracked if name}
assert EXPECTED and any(n.startswith('solution/') for n in EXPECTED)
baseline = json.loads((ROOT / 'BASELINE_PROVENANCE.json').read_text())
official = {n:h for n,h in baseline['source_sha256'].items() if n.startswith(('harness/', 'task/', 'tests/'))}
assert len(official) == 15
assert all(EXPECTED.get(n) == h for n,h in official.items()), 'Official task/harness/tests changed.'
provenance = {'repository': REPOSITORY, 'requested_ref': REF, 'commit': COMMIT,
              'source_sha256': EXPECTED, 'training_requested': RUN_TRAINING}
(WORKSPACE / 'checkout-provenance.json').write_text(json.dumps(provenance, indent=2))
print('Resolved source:', COMMIT)
print('Runtime source files:', len(EXPECTED))
print('Training enabled:', RUN_TRAINING)


In [ ]:
if RUN_TRAINING:
    gpus = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
    print(gpus)
    assert 'T4' in gpus, 'Select Tesla T4 for comparable measurements.'
    run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao', 'xformers', 'flash-attn', 'liger-kernel', 'unsloth', 'unsloth-zoo'])
    run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.11.0', '--index-url', 'https://download.pytorch.org/whl/cu128'])
    run([sys.executable, '-m', 'pip', 'install', '-q', 'triton==3.6.0', 'soup-cli[train]==0.77.0', 'transformers==5.16.1', 'peft==0.20.0', 'trl==0.29.1', 'bitsandbytes==0.50.2', 'accelerate==1.14.0', 'datasets==4.8.5', 'pytest'])
    run([sys.executable, 'task/prepare_data.py'], cwd=ROOT)
    env = os.environ.copy(); env['SOUP_APPLY_SOLUTION'] = '1'
    with (ROOT / 'qualification-tests.log').open('w') as log:
        tested = subprocess.run([sys.executable, '-m', 'pytest', 'solution', 'tests', '-q', '--tb=short'], cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT)
    print((ROOT / 'qualification-tests.log').read_text()[-6000:])
    assert tested.returncode == 0, 'Tests failed; inspect qualification-tests.log.'
else:
    print('Checkout verified. Enable T4 and RUN_TRAINING in a fresh session to train.')


In [ ]:
if RUN_TRAINING:
    # Original full cold ABBA: base, solution, solution, base.
    run([sys.executable, 'harness/run_task.py', '--repeats', '2'], cwd=ROOT)


In [ ]:
if RUN_TRAINING:
    checks = subprocess.run([sys.executable, 'harness/check_rules.py'], cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (ROOT / 'qualification-checks.log').write_text(checks.stdout)
    print(checks.stdout)
    score = subprocess.run([sys.executable, 'harness/score.py', 'runs/result.json'], cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (ROOT / 'qualification-score.log').write_text(score.stdout)
    print(score.stdout)
    after = {n:digest(ROOT / n) for n in EXPECTED}
    assert after == EXPECTED, 'Source changed during the run.'
    provenance.update(source_unchanged=True, check_rules_exit_code=checks.returncode, score_exit_code=score.returncode,
                      classification='New experimental run; inspect raw checks/results, do not reuse historical baseline claims.')
    (ROOT / 'QUALIFICATION_PROVENANCE.json').write_text(json.dumps(provenance, indent=2))


In [ ]:
if RUN_TRAINING:
    import zipfile
    out = WORKSPACE / f'soup-lab-{COMMIT[:12]}-evidence.zip'
    files = {n:ROOT / n for n in EXPECTED}
    for p in (ROOT / 'runs').rglob('*'):
        if not p.is_file() or any(x in ('triton_cache', 'inductor_cache', '__pycache__') for x in p.relative_to(ROOT).parts):
            continue
        if p.suffix in ('.json', '.jsonl', '.yaml', '.log') and not p.name.startswith('tokenizer'):
            files[p.relative_to(ROOT).as_posix()] = p
    for n in ('qualification-tests.log', 'qualification-checks.log', 'qualification-score.log', 'QUALIFICATION_PROVENANCE.json'):
        files[n] = ROOT / n
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        hashes = {}
        for name,p in sorted(files.items()):
            data=p.read_bytes(); z.writestr(name,data); hashes[name]=hashlib.sha256(data).hexdigest()
        z.writestr('MANIFEST.json',json.dumps(hashes,indent=2))
    print('Evidence:', out, digest(out))
    assert checks.returncode == 0, 'Official checks failed; evidence preserved.'
    assert score.returncode == 0
